# New telemetry: selected-site flattening and EDA

Methodological mirror of `split_and_flatten_selected_site_stats.ipynb`. The compatible `six_month` filenames are retained, while the plots and summaries report the actual new-data range.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

SITE_STATS_PATHS = [
    PROJECT_DIR / "data" / "raw" / "site_stats_part1.csv",
    PROJECT_DIR / "data" / "raw" / "site_stats_part2.csv",
    PROJECT_DIR / "data" / "raw" / "site_stats_part3.csv",
]
ALARM_LOGS_PATH = PROJECT_DIR / "data" / "raw" / "vrm_alarm_logs_clean.csv"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "config" / "sites.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Copy config/sites.example.csv to config/sites.csv before running."
    )


PROJECT_DIR = find_project_root()
SITES_CONFIG = PROJECT_DIR / "config" / "sites.csv"
sites_config = pd.read_csv(SITES_CONFIG)
sites_config["site_id"] = pd.to_numeric(sites_config["site_id"], errors="raise").astype(int)
SELECTED_SITES = dict(zip(sites_config["site_id"], sites_config["site_name"]))

ATTRIBUTE_RENAMES = {
    "gc": "Gc",
    "gb": "Gb",
}

ATTRIBUTE_TO_METRIC = {
    "Pc": "Solar -> Load (kWh)",
    "Pb": "Solar -> Battery (kWh)",
    "Pg": "Solar -> Grid (kWh)",
    "Bc": "Battery -> Load (kWh)",
    "Bg": "Battery -> Grid (kWh)",
    "Gc": "Grid -> Load (kWh)",
    "Gb": "Grid -> Battery (kWh)",
    "kwh": "Total kWh",
    "bs": "Battery SoC (%)",
}

METRIC_COLUMNS = [
    "Solar yield (kWh)",
    "Total solar yield (kWh)",
    "Consumption (kWh)",
    "Solar -> Load (kWh)",
    "Solar -> Battery (kWh)",
    "Solar -> Grid (kWh)",
    "Battery -> Load (kWh)",
    "Battery -> Grid (kWh)",
    "Grid -> Load (kWh)",
    "Grid -> Battery (kWh)",
    "Total kWh",
    "Solar irradiance (W/m2)",
    "Solar irradiance forecast",
    "PV charger yield forecast",
    "PV inverter yield forecast",
    "Solar yield forecast",
    "Battery SoC (%)",
]

DAILY_AGGREGATION = {
    "Solar yield (kWh)": "sum",
    "Total solar yield (kWh)": "sum",
    "Consumption (kWh)": "mean",
    "Solar -> Load (kWh)": "sum",
    "Solar -> Battery (kWh)": "sum",
    "Solar -> Grid (kWh)": "sum",
    "Battery -> Load (kWh)": "sum",
    "Battery -> Grid (kWh)": "sum",
    "Grid -> Load (kWh)": "sum",
    "Grid -> Battery (kWh)": "sum",
    "Total kWh": "sum",
    "Solar irradiance (W/m2)": "mean",
    "Solar irradiance forecast": "mean",
    "PV charger yield forecast": "sum",
    "PV inverter yield forecast": "sum",
    "Solar yield forecast": "sum",
    "Battery SoC (%)": "mean",
}

print("Stats paths:")
for path in SITE_STATS_PATHS:
    print(" -", path)
print("Alarm path:", ALARM_LOGS_PATH)
print("Output directory:", OUTPUT_DIR)
print("All stats parts exist:", all(path.exists() for path in SITE_STATS_PATHS))
print("Alarms exist:", ALARM_LOGS_PATH.exists())

parameter_mapping = pd.DataFrame(
    ATTRIBUTE_TO_METRIC.items(), columns=["attribute", "output_column"]
)
parameter_mapping.to_csv(OUTPUT_DIR / "vrm_parameter_mapping.csv", index=False)

## Load, concatenate, validate, and select the 12 sites

In [ ]:
selected_parts = []
part_summary_rows = []
usecols = ["site_id", "site_name", "attribute", "label", "timestamp_utc", "value"]
expected_columns = set(usecols)

for input_path in SITE_STATS_PATHS:
    part_rows = 0
    selected_rows = 0
    for chunk in pd.read_csv(input_path, usecols=usecols, chunksize=300_000):
        part_rows += len(chunk)
        selected = chunk[chunk["site_id"].isin(SELECTED_SITES)].copy()
        selected_rows += len(selected)
        if not selected.empty:
            selected["source_file"] = input_path.name
            selected_parts.append(selected)
    part_summary_rows.append({
        "input_file": input_path.name,
        "rows_read": part_rows,
        "selected_site_rows": selected_rows,
        "file_size_bytes": input_path.stat().st_size,
    })

site_stats_selected = pd.concat(selected_parts, ignore_index=True)
del selected_parts

raw_timestamps = pd.to_datetime(
    site_stats_selected["timestamp_utc"], utc=True, errors="coerce"
)
raw_numeric_values = pd.to_numeric(site_stats_selected["value"], errors="coerce")
invalid_timestamp_rows = int(raw_timestamps.isna().sum())
non_numeric_value_rows = int(
    (site_stats_selected["value"].notna() & raw_numeric_values.isna()).sum()
)
raw_duplicate_records = int(
    site_stats_selected.duplicated(subset=usecols, keep=False).sum()
)

site_stats_selected["attribute"] = site_stats_selected["attribute"].replace(
    ATTRIBUTE_RENAMES
)
site_stats_selected["parameter"] = site_stats_selected["attribute"].map(
    ATTRIBUTE_TO_METRIC
)
site_stats_selected["timestamp_utc"] = raw_timestamps
site_stats_selected["value"] = raw_numeric_values

unmapped_attributes = sorted(
    site_stats_selected.loc[
        site_stats_selected["parameter"].isna(), "attribute"
    ].dropna().unique()
)
normalized_parameter_key_duplicates = int(
    site_stats_selected.dropna(subset=["timestamp_utc", "parameter"])
    .duplicated(subset=["site_id", "timestamp_utc", "parameter"], keep=False)
    .sum()
)

validation_site_rows = []
expected_attributes = set(ATTRIBUTE_TO_METRIC)
for site_id in SELECTED_SITES:
    site_raw = site_stats_selected[site_stats_selected["site_id"] == site_id]
    available = set(site_raw["attribute"].dropna().unique())
    validation_site_rows.append({
        "site_id": site_id,
        "site_name": SELECTED_SITES[site_id],
        "input_rows": len(site_raw),
        "invalid_timestamp_rows": int(site_raw["timestamp_utc"].isna().sum()),
        "non_numeric_nonblank_values": int(
            (site_raw["value"].isna() & site_raw["label"].notna()).sum()
        ),
        "battery_soc_source_rows": int((site_raw["attribute"] == "bs").sum()),
        "battery_soc_source_non_null": int(
            site_raw.loc[site_raw["attribute"] == "bs", "value"].notna().sum()
        ),
        "available_normalized_attributes": " | ".join(sorted(available)),
        "missing_mapped_attributes": " | ".join(sorted(expected_attributes - available)),
        "unmapped_attributes": " | ".join(
            sorted(site_raw.loc[site_raw["parameter"].isna(), "attribute"].dropna().unique())
        ),
        "raw_range_start": site_raw["timestamp_utc"].min(),
        "raw_range_end": site_raw["timestamp_utc"].max(),
    })

part_summary = pd.DataFrame(part_summary_rows)
input_validation_by_site = pd.DataFrame(validation_site_rows)
input_validation = pd.DataFrame([{
    "parts_read": len(SITE_STATS_PATHS),
    "selected_rows": len(site_stats_selected),
    "selected_sites_found": site_stats_selected["site_id"].nunique(),
    "invalid_timestamp_rows": invalid_timestamp_rows,
    "non_numeric_nonblank_values": non_numeric_value_rows,
    "raw_duplicate_rows": raw_duplicate_records,
    "normalized_parameter_duplicate_rows": normalized_parameter_key_duplicates,
    "unmapped_attributes": " | ".join(unmapped_attributes),
}])

assert len(part_summary) == 3
assert set(site_stats_selected["site_id"].dropna().astype(int).unique()) == set(SELECTED_SITES)
assert (input_validation_by_site["battery_soc_source_non_null"] > 0).all()

part_summary.to_csv(OUTPUT_DIR / "new_telemetry_part_summary.csv", index=False)
input_validation.to_csv(OUTPUT_DIR / "new_telemetry_input_validation.csv", index=False)
input_validation_by_site.to_csv(
    OUTPUT_DIR / "new_telemetry_site_validation.csv", index=False
)

site_stats_selected.dropna(subset=["timestamp_utc", "parameter"], inplace=True)

print("Selected rows before mapped-parameter filtering:", f"{input_validation.iloc[0]['selected_rows']:,}")
print("Selected rows used by the reference mapping:", f"{len(site_stats_selected):,}")
print("Selected sites found:", site_stats_selected["site_id"].nunique())
print("Available mapped parameters:", sorted(site_stats_selected["parameter"].unique()))
print("Unmapped new-data attributes retained in validation only:", unmapped_attributes)
print("Normalized parameter-key duplicate rows:", normalized_parameter_key_duplicates)
display(part_summary)
display(input_validation)
display(input_validation_by_site)


## Create one wide dataframe and CSV per site

This uses the same `pivot_table` pattern supplied in the request: one row per site and timestamp. The pivot uses the actual VRM parameter code because the source `label` field repeats generic labels with the same value and would create false metrics.

In [ ]:
def pivot_parameter_columns(df):
    wide = df.pivot_table(
        index=["site_id", "timestamp_utc"],
        columns="parameter",
        values="value",
        aggfunc="first",
    ).reset_index()
    wide.columns.name = None

    # Keep the same metric layout for every site, even when a metric is absent.
    return wide.reindex(columns=["site_id", "timestamp_utc", *METRIC_COLUMNS])


site_wide_frames = {}

for site_id in SELECTED_SITES:
    site_df = site_stats_selected[site_stats_selected["site_id"] == site_id].copy()
    globals()[f"site_stats_{site_id}"] = site_df

    site_wide = pivot_parameter_columns(site_df)
    site_wide.sort_values("timestamp_utc", inplace=True)
    site_wide.reset_index(drop=True, inplace=True)

    globals()[f"site_stats_{site_id}_wide"] = site_wide
    site_wide_frames[site_id] = site_wide

    output_path = OUTPUT_DIR / f"site_{site_id}_six_month_vrm.csv"
    site_wide.to_csv(output_path, index=False)
    print(site_id, site_wide.shape, "->", output_path.name)

next(iter(site_wide_frames.values())).head()

## Battery SOC mapping verification

In [ ]:
soc_mapping_rows = []
for site_id, wide in site_wide_frames.items():
    soc = pd.to_numeric(wide["Battery SoC (%)"], errors="coerce")
    soc_mapping_rows.append({
        "site_id": site_id,
        "battery_soc_non_null_rows": int(soc.notna().sum()),
        "battery_soc_min": soc.min(),
        "battery_soc_max": soc.max(),
    })

battery_soc_mapping_verification = pd.DataFrame(soc_mapping_rows)
assert (battery_soc_mapping_verification["battery_soc_non_null_rows"] > 0).all()
battery_soc_mapping_verification.to_csv(
    OUTPUT_DIR / "battery_soc_mapping_verification.csv", index=False
)
battery_soc_mapping_verification


## EDA summaries

As in the reference notebook, each wide dataframe is indexed by UTC timestamp, converted to numeric metrics, checked for missing values and sampling gaps, and resampled into daily aggregates.

In [ ]:
indexed_site_frames = {}
daily_by_site = {}
site_summary_rows = []
metric_summary_parts = []
gap_summary_rows = []

for site_id, wide in site_wide_frames.items():
    indexed = wide.copy()
    indexed["timestamp_utc"] = pd.to_datetime(indexed["timestamp_utc"], utc=True, errors="coerce")
    indexed.dropna(subset=["timestamp_utc"], inplace=True)
    indexed.set_index("timestamp_utc", inplace=True)
    indexed.sort_index(inplace=True)

    for column in METRIC_COLUMNS:
        indexed[column] = pd.to_numeric(indexed[column], errors="coerce")

    indexed_site_frames[site_id] = indexed
    globals()[f"site_stats_{site_id}_indexed"] = indexed

    metric_summary = indexed[METRIC_COLUMNS].describe().T
    metric_summary["missing_rows"] = indexed[METRIC_COLUMNS].isna().sum()
    metric_summary["missing_pct"] = (
        metric_summary["missing_rows"] / len(indexed) * 100
    ).round(2)
    metric_summary["non_null_rows"] = indexed[METRIC_COLUMNS].notna().sum()
    metric_summary.insert(0, "site_id", site_id)
    metric_summary.insert(1, "metric", metric_summary.index)
    metric_summary_parts.append(metric_summary.reset_index(drop=True))

    deltas = indexed.index.to_series().diff().dropna()
    large_gaps = deltas[deltas > pd.Timedelta(minutes=20)]
    common_interval = deltas.mode().iloc[0] if not deltas.empty else pd.NaT

    site_summary_rows.append({
        "site_id": site_id,
        "site_name": SELECTED_SITES[site_id],
        "rows": len(indexed),
        "range_start": indexed.index.min(),
        "range_end": indexed.index.max(),
        "common_interval": common_interval,
        "gaps_over_20_minutes": len(large_gaps),
    })

    for gap_end, gap in large_gaps.items():
        gap_summary_rows.append({
            "site_id": site_id,
            "previous_timestamp": gap_end - gap,
            "next_timestamp": gap_end,
            "gap": gap,
        })

    daily = indexed[METRIC_COLUMNS].resample("D").agg(DAILY_AGGREGATION)
    daily_by_site[site_id] = daily
    globals()[f"site_stats_{site_id}_daily"] = daily

site_summary = pd.DataFrame(site_summary_rows)
metric_summary_all_sites = pd.concat(metric_summary_parts, ignore_index=True)
gap_summary = pd.DataFrame(gap_summary_rows)

site_summary.to_csv(OUTPUT_DIR / "selected_sites_eda_summary.csv", index=False)
metric_summary_all_sites.to_csv(OUTPUT_DIR / "selected_sites_metric_summary.csv", index=False)
gap_summary.to_csv(OUTPUT_DIR / "selected_sites_timestamp_gaps.csv", index=False)

site_summary

## Telemetry coverage across the actual new-data window

In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

INTENDED_START = pd.to_datetime(site_summary["range_start"], utc=True).min()
INTENDED_END = pd.to_datetime(site_summary["range_end"], utc=True).max()

SITE_DISPLAY_NAMES = SELECTED_SITES
SITE_ORDER = list(SITE_DISPLAY_NAMES)

coverage_plot_data = site_summary.set_index("site_id").loc[SITE_ORDER].copy()
coverage_plot_data["range_start"] = pd.to_datetime(
    coverage_plot_data["range_start"], utc=True
)
coverage_plot_data["range_end"] = pd.to_datetime(
    coverage_plot_data["range_end"], utc=True
)
coverage_plot_data["full_intended_coverage"] = (
    coverage_plot_data["range_start"].le(INTENDED_START)
    & coverage_plot_data["range_end"].ge(INTENDED_END)
)

FULL_COLOR = "#2f8f8b"
PARTIAL_COLOR = "#2878b8"

fig, axis = plt.subplots(figsize=(16, 10))
axis.axvspan(
    INTENDED_START,
    INTENDED_END,
    color="#e8f0f7",
    alpha=0.72,
    zorder=0,
)

for y_position, (site_id, row) in enumerate(coverage_plot_data.iterrows()):
    color = FULL_COLOR if row["full_intended_coverage"] else PARTIAL_COLOR
    axis.plot(
        [row["range_start"], row["range_end"]],
        [y_position, y_position],
        color=color,
        linewidth=8,
        solid_capstyle="round",
        zorder=3,
    )

axis.set_yticks(range(len(coverage_plot_data)))
axis.set_yticklabels(
    [SITE_DISPLAY_NAMES[site_id] for site_id in coverage_plot_data.index],
    fontsize=11,
    fontweight="semibold",
    color="#374151",
)
axis.invert_yaxis()

axis.xaxis.set_major_locator(mdates.MonthLocator())
axis.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
month_ticks = pd.date_range(
    INTENDED_START.normalize().replace(day=1),
    INTENDED_END.normalize().replace(day=1),
    freq="MS",
)
axis.set_xticks(month_ticks)
axis.set_xlim(
    INTENDED_START - pd.Timedelta(days=2),
    INTENDED_END + pd.Timedelta(days=2),
)
axis.tick_params(axis="x", labelsize=15, width=1.8, length=8, pad=10)
for label in axis.get_xticklabels():
    label.set_fontweight("bold")
    label.set_color("#1f2937")

axis.set_xlabel(
    "Telemetry date (UTC)",
    fontsize=17,
    fontweight="bold",
    color="#111827",
    labelpad=14,
)
axis.set_title(
    "Telemetry Coverage for the 12 Selected Sites",
    fontsize=28,
    fontweight="heavy",
    color="#111827",
    pad=34,
)
axis.text(
    0.5,
    1.025,
    f"Actual analysis window: {INTENDED_START:%d %b %Y} - {INTENDED_END:%d %b %Y}",
    transform=axis.transAxes,
    ha="center",
    va="bottom",
    fontsize=15,
    fontweight="bold",
    color=PARTIAL_COLOR,
)

axis.grid(axis="x", color="#cbd5e1", linewidth=1, alpha=0.8)
axis.grid(axis="y", visible=False)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.spines["bottom"].set_color("#94a3b8")
axis.spines["left"].set_color("#94a3b8")
axis.margins(x=0.01, y=0.08)

legend_handles = [
    Line2D([0], [0], color=FULL_COLOR, linewidth=8, label="Covers the full intended window"),
    Line2D([0], [0], color=PARTIAL_COLOR, linewidth=8, label="Partial intended-window coverage"),
]
legend = axis.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.14),
    ncol=2,
    frameon=False,
    fontsize=13,
    handlelength=2.8,
    columnspacing=4,
)
for text in legend.get_texts():
    text.set_fontweight("semibold")

fig.tight_layout(rect=[0.01, 0.06, 0.99, 0.92])
coverage_plot_path = OUTPUT_DIR / "selected_sites_telemetry_coverage.png"
fig.savefig(coverage_plot_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved coverage plot:", coverage_plot_path)
display(
    coverage_plot_data[
        ["range_start", "range_end", "full_intended_coverage"]
    ].reset_index()
)


## Site overview plot

Change `SITE_ID` to inspect another selected site. Only metrics containing data are plotted.

In [ ]:
SITE_ID = 100001

if plt is None:
    print("Install matplotlib to display the site overview plot.")
else:
    plot_df = indexed_site_frames[SITE_ID]
    available_metrics = [column for column in METRIC_COLUMNS if plot_df[column].notna().any()]

    fig, axes = plt.subplots(
        len(available_metrics), 1,
        figsize=(15, 3 * len(available_metrics)),
        sharex=True,
    )
    if len(available_metrics) == 1:
        axes = [axes]

    for axis, metric in zip(axes, available_metrics):
        plot_df[metric].plot(ax=axis, linewidth=0.8)
        axis.set_title(metric)
        axis.grid(True, alpha=0.25)

    fig.suptitle(f"Site {SITE_ID}: six-month VRM overview", y=1.01)
    fig.tight_layout()
    plt.show()